# 01. Context 分支训练（FP32）

本课程把真实 sEMG 手势识别项目的三个分支（Context / Hybrid / Delay-SNN）各自拆成
"训练"+"量化"两个 notebook，最后两个 notebook 做融合。这是第一个：**Context 分支**
（`ClassAdaptiveContextSNN`）从零训练的完整过程，用真实的 NinaPro DB5 数据和真实项目代码
（`training/semg_snn_90_loop/`）。

概念性背景（LIF 神经元、替代梯度、训练工程细节）见配套 md 教程
[01~03 章](../01-snn-basics.md)，这里只聚焦"这个具体项目实际是怎么训的"。

## 协议(与 README.md 一致)

| 项目 | 取值 |
|---|---|
| 数据集 | NinaPro DB5, Exercise A |
| 类别 | 13 类（Rest + 12 个手指动作）|
| 采样率 / 通道 | 200 Hz / 16 通道 |
| 窗口 / 步长 | 100 点(500ms) / 20 点(100ms)，80% 重叠 |
| train / val / test repetition | 1,2,4,6 / 3 / 5 |
| 样本数 | 44,630 / 11,060 / 11,276 |

**运行要求**：需要 GPU，需要 `data/` 目录下已经跑过 `prepare.py` 和
`prepare_continuous_context.py` 生成好的 `.npz` 数据文件（本机已经生成好）。

In [ ]:
import sys, json, random
from pathlib import Path

import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

PROJECT_ROOT = Path("training/semg_snn_90_loop")
sys.path.insert(0, str(PROJECT_ROOT))

from train import EMGDataset          # 真实项目的数据集类，直接复用
from model import ClassAdaptiveContextSNN  # 真实项目的 Context 分支模型

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

# notebook 训练产物单独存放，不覆盖项目原有的 runs/ 结果
NB_RUNS = PROJECT_ROOT / "runs_notebook"
NB_RUNS.mkdir(exist_ok=True)

## 1. 数据：`EMGDataset` 的 context / continuous_context / stream_context

`EMGDataset`（定义在 `train.py`）不只是简单地读一个窗口的特征，Context 分支需要
"过去若干个窗口拼起来的历史"，这由三个参数控制：

- `context=23`：每个样本带 23 个连续窗口的历史（23×100ms ≈ 2.2 秒的上下文）。
- `continuous_context=True`：历史特征从预先算好的 `continuous_features.npy` 里按
  真实时间顺序读取（而不是从当前 split 内部按索引回溯，那样在 split 边界会出错）。
- `stream_context=True`：**完全因果**——历史索引只根据"同一受试者、时间上更早"来选取，
  不查询 repetition 编号，也不会因为动作切换而重置。这是最贴近"真实在线部署"的条件
  （模型部署后不可能提前知道动作分段边界）。

两阶段训练用的是不同的 `continuous_context`/`stream_context` 组合，下面会具体展开。

In [ ]:
def load_datasets(context: int, continuous_context: bool, stream_context: bool):
    data = PROJECT_ROOT / "data"
    sets = {
        split: EMGDataset(
            data / f"{split}.npz", data / "normalization.npz", split == "train",
            context=context, continuous_context=continuous_context, stream_context=stream_context,
        )
        for split in ("train", "val", "test")
    }
    for split, ds in sets.items():
        print(f"{split}: {len(ds)} samples, feature dim={ds.features.shape[1]}")
    return sets

# 先看一眼 stage-2（完全因果流式）配置下的数据规模
_preview = load_datasets(context=23, continuous_context=True, stream_context=True)
counts = np.bincount(_preview["test"].y, minlength=13)
print("test 集 Rest 占比:", counts[0] / counts.sum())

## 2. 模型：`ClassAdaptiveContextSNN`

结构（详见 `model.py`）：

```
encoder = Linear(336, 512) -> LayerNorm(512)      # 每个窗口独立编码
LIF 层1（512 神经元，PLIF 逐神经元可学习衰减 beta1 + beta1_offset）
  -> fc2 = Linear(512, 256) -> LayerNorm(256)
LIF 层2（256 神经元，PLIF 逐神经元可学习衰减 beta2 + beta2_offset）
  -> out = Linear(256, 13)
```

和教程 03 章的两层 SNN 相比，多了两个关键设计：

1. **PLIF（Parametric LIF）**：衰减系数不是全局共享一个标量，而是
   `beta + offset`，`offset` 是逐神经元的参数，让不同神经元学到不同的记忆时长。
2. **逐类别自适应读出衰减**（`context_gamma_logit`，13 维，每类一个）：
   越晚的窗口对最终 logits 贡献的权重是 `gamma_class ** (剩余窗口数)`，
   不同手势类别可以学到不同的"该往回看多久"。

In [ ]:
model = ClassAdaptiveContextSNN(features=_preview["train"].features.shape[1]).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f"Context 模型参数量: {n_params:,}")
del _preview  # 只是用来看一眼数据规模,训练时会重新构造 DataLoader

## 3. 训练循环

和教程 03 章一样：AdamW + 余弦学习率、类别加权交叉熵（`weight_power` 控制加权强度，
缓解 Rest 类占 63% 的不均衡）、label smoothing、梯度裁剪、早停 + 验证集选 checkpoint。
和教程不同的是**两阶段热启动**——这是真实项目实际使用的策略，README 里写的
命令就是 stage 2：

```bash
python train.py --model context_class_adaptive --context 23 \
  --continuous-context --stream-context \
  --epochs 24 --patience 7 --batch-size 256 --lr 0.0002 \
  --run-name context23_class_plif_stream \
  --init runs/context23_class_plif_continuous/best.pt
```

**为什么要两阶段**：直接从零在"完全因果流式"（stream_context=True）条件下训练，
优化问题更难——每个样本的历史长度、分布都更不规则。项目实际做法是先在一个更规整的
条件下（`continuous_context=True, stream_context=False`，历史仍然连续，但知道
repetition 边界，不会跨动作段乱采样历史）训练出一个较好的 stage-1 模型，
再用它热启动，微调到完全因果的 stream 条件——这是一种"课程学习"（curriculum）思路：
先学简单条件，再迁移到难条件。

> 说明：真实项目当时 stage-1 的热启动链条更长（`context23_class_plif_continuous`
> 又是从更早的 `context23_adaptive_continuous` checkpoint 热启动的），这里为了
> 教学简洁，stage-1 从随机初始化直接训练，只保留"两阶段课程学习"这个核心结构。
> stage-2 的超参数是 README 里记录的确切命令。

In [ ]:
def train_context_model(run_name: str, epochs: int, lr: float, patience: int,
                          continuous_context: bool, stream_context: bool,
                          weight_power: float = 0.25, label_smoothing: float = 0.02,
                          init_checkpoint: Path | None = None, batch_size: int = 256,
                          seed: int = 42):
    """Context 分支训练循环,逻辑和 train.py 的 main() 一致,包成函数方便 notebook 里复用。"""
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)

    sets = {
        split: EMGDataset(
            PROJECT_ROOT / "data" / f"{split}.npz", PROJECT_ROOT / "data" / "normalization.npz",
            split == "train", context=23,
            continuous_context=continuous_context, stream_context=stream_context,
        )
        for split in ("train", "val", "test")
    }
    loaders = {
        "train": DataLoader(sets["train"], batch_size, shuffle=True, num_workers=4, pin_memory=True),
        "val": DataLoader(sets["val"], batch_size * 2, num_workers=4, pin_memory=True),
        "test": DataLoader(sets["test"], batch_size * 2, num_workers=4, pin_memory=True),
    }
    feature_count = sets["train"].features.shape[1]
    model = ClassAdaptiveContextSNN(feature_count).to(device)

    if init_checkpoint is not None:
        source = torch.load(init_checkpoint, map_location=device, weights_only=False)["model"]
        compatible = {k: v for k, v in source.items()
                      if k in model.state_dict() and model.state_dict()[k].shape == v.shape}
        missing, unexpected = model.load_state_dict(compatible, strict=False)
        print(f"warm-start from {init_checkpoint}: loaded={len(compatible)} missing={missing}")

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=2e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, epochs)
    counts = np.bincount(sets["train"].y, minlength=13)
    class_weights = torch.tensor((counts.sum() / (13 * counts)) ** weight_power,
                                  dtype=torch.float32, device=device)

    run_dir = NB_RUNS / run_name
    run_dir.mkdir(parents=True, exist_ok=True)
    best_acc, stale = -1.0, 0

    for epoch in range(1, epochs + 1):
        model.train()
        losses = []
        for f, raw, y, subject in loaders["train"]:
            f, raw, y, subject = f.to(device), raw.to(device), y.to(device), subject.to(device)
            optimizer.zero_grad(set_to_none=True)
            output, _ = model(f, raw, subject)
            loss = nn.functional.cross_entropy(output, y, weight=class_weights,
                                                label_smoothing=label_smoothing)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 2.0)
            optimizer.step()
            losses.append(loss.item())
        scheduler.step()

        val_metrics = evaluate_context(model, loaders["val"])
        print(f"epoch {epoch:02d}  loss={np.mean(losses):.4f}  "
              f"val_acc={val_metrics['accuracy']:.4f}  val_macro_f1={val_metrics['macro_f1']:.4f}")

        if val_metrics["accuracy"] > best_acc:
            best_acc, stale = val_metrics["accuracy"], 0
            torch.save({"model": model.state_dict(), "epoch": epoch, "validation": val_metrics},
                       run_dir / "best.pt")
        else:
            stale += 1
            if stale >= patience:
                print("early stopping")
                break

    ckpt = torch.load(run_dir / "best.pt", map_location=device, weights_only=False)
    model.load_state_dict(ckpt["model"])
    test_metrics = evaluate_context(model, loaders["test"])
    print(f"\n=== {run_name} 最终结果（第 {ckpt['epoch']} 轮 checkpoint）===")
    print(f"val:  accuracy={ckpt['validation']['accuracy']:.4f}  macro_f1={ckpt['validation']['macro_f1']:.4f}")
    print(f"test: accuracy={test_metrics['accuracy']:.4f}  macro_f1={test_metrics['macro_f1']:.4f}  "
          f"gesture_accuracy={test_metrics['gesture_accuracy']:.4f}")
    return run_dir / "best.pt", test_metrics


@torch.no_grad()
def evaluate_context(model, loader):
    model.eval()
    preds, targets = [], []
    for f, raw, y, subject in loader:
        output, _ = model(f.to(device), raw.to(device), subject.to(device))
        preds.extend(output.argmax(1).cpu().tolist())
        targets.extend(y.tolist())
    y_arr, p_arr = np.asarray(targets), np.asarray(preds)
    return {
        "accuracy": accuracy_score(y_arr, p_arr),
        "macro_f1": f1_score(y_arr, p_arr, average="macro"),
        "gesture_accuracy": float(np.mean(p_arr[y_arr != 0] == y_arr[y_arr != 0])),
    }

### Stage 1：连续历史 + 已知 repetition 边界（更容易优化的条件）

首次运行建议先把 `epochs` 调小（比如 3~5）做一次 smoke test，确认代码能跑通、
loss 在下降，再跑完整的 epoch 数。下面给的是与真实项目同一量级的完整超参数。

In [ ]:
stage1_checkpoint, stage1_test_metrics = train_context_model(
    run_name="context23_stage1_continuous_nb",
    epochs=32, lr=3e-4, patience=9,
    continuous_context=True, stream_context=False,
    weight_power=0.25, label_smoothing=0.02,
    init_checkpoint=None,
)

### Stage 2：完全因果流式（部署时真实面对的条件），从 stage 1 热启动

这一段的超参数和 README 里记录的真实命令完全一致：`epochs=24, lr=2e-4, patience=7`。

In [ ]:
stage2_checkpoint, stage2_test_metrics = train_context_model(
    run_name="context23_stage2_stream_nb",
    epochs=24, lr=2e-4, patience=7,
    continuous_context=True, stream_context=True,
    weight_power=0.25, label_smoothing=0.02,
    init_checkpoint=stage1_checkpoint,
)

## 4. 结果对照

真实项目里这条 stream 条件下的 Context-23 单分支测试结果是
**accuracy 88.53% / macro-F1 0.7822 / gesture-accuracy 73.76%**（`RESULTS.md`，
"完全无边界"协议下的 "Class-PLIF Context-23" 行）。你跑出来的数字应该在这个量级附近
（不会完全相同：随机种子相同，但 stage-1 的热启动链条比真实项目短一步,所以两边的
优化轨迹不会逐位重合）。

**单个 Context 分支到不了 90%**——这是预期之内的,不是训练没调好。真实项目里
90%+ 的结果来自 Context + Hybrid + Delay-SNN **三分支融合**,这是 [07_fusion_fp32.ipynb](07_fusion_fp32.ipynb)
要做的事。

In [ ]:
print("Stage 2（完全因果流式）测试结果:")
for k, v in stage2_test_metrics.items():
    print(f"  {k}: {v:.4f}")
print("\n参考值(真实项目, RESULTS.md 完全无边界协议, Class-PLIF Context-23 单分支):")
print("  accuracy: 0.8853  macro_f1: 0.7822  gesture_accuracy: 0.7376")

## 下一步

打开 [02_context_quantization.ipynb](02_context_quantization.ipynb)，把这个 FP32
checkpoint 改造成硬件友好、可量化部署的版本（QAT）。三个分支的训练+量化都做完之后，
[07_fusion_fp32.ipynb](07_fusion_fp32.ipynb) 和
[08_fusion_hw_qat.ipynb](08_fusion_hw_qat.ipynb) 会把它们融合起来。